# 3. Survival Analysis

Kaplan-Meier survival curves and log-rank tests stratified by phenotype.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import logrank_oneway
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data (from notebook 2)
df = pd.read_csv('LiverMets_Final_Dataset.csv')

complete_tnm = df[
    (df['T_STAGE'].notna()) & (df['T_STAGE'] != 'ND') &
    (df['N_STAGE'].notna()) & (df['N_STAGE'] != 'ND') &
    (df['M_STAGE'].notna()) & (df['M_STAGE'] != 'ND')
]
included = complete_tnm[
    (complete_tnm['SURVIVAL_YEARS'].notna()) & 
    (complete_tnm['SURVIVAL_YEARS'] > 0) &
    (complete_tnm['VITAL_STATUS'].notna())
]

# Define phenotypes
df_ph = included.copy()
df_ph['PHENOTYPE'] = np.nan
mask1 = (df_ph['M_STAGE'] == 'M0') & (df_ph['N_STAGE'].isin(['N0', 'N1']))
df_ph.loc[mask1, 'PHENOTYPE'] = 1
mask2a = (df_ph['M_STAGE'] == 'M0') & (df_ph['N_STAGE'] == 'N2')
mask2b = (df_ph['M_STAGE'] == 'M1') & (df_ph['N_STAGE'].isin(['N0', 'N1']))
df_ph.loc[mask2a | mask2b, 'PHENOTYPE'] = 2
mask3 = (df_ph['M_STAGE'] == 'M1') & (df_ph['N_STAGE'] == 'N2')
df_ph.loc[mask3, 'PHENOTYPE'] = 3

print(f"Cohort: {len(df_ph):,} patients")

## Kaplan-Meier Survival Curves

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# Overall survival by phenotype
fig, ax = plt.subplots(figsize=(12, 7))

kmf = KaplanMeierFitter()
phenotype_labels = {1: 'Favourable', 2: 'Intermediate', 3: 'Adverse'}
colors = ['#2ecc71', '#f39c12', '#e74c3c']

for ph, color, label in zip([1, 2, 3], colors, ['Favourable', 'Intermediate', 'Adverse']):
    cohort = df_ph[df_ph['PHENOTYPE'] == ph]
    kmf.fit(cohort['SURVIVAL_YEARS'], cohort['VITAL_STATUS'], label=f'{label} (n={len(cohort):,})')
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2.5)

ax.set_xlabel('Time (years)', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability of Survival', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=11)
ax.set_title('Overall Survival by CART Phenotype', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Log-Rank Tests

In [ ]:
# Pairwise log-rank tests
print("Log-Rank Tests (Overall Survival by Phenotype)")
print("=" * 70)

phenotype_pairs = [(1, 2), (1, 3), (2, 3)]
phenotype_names = {1: 'Favourable', 2: 'Intermediate', 3: 'Adverse'}

for ph1, ph2 in phenotype_pairs:
    cohort1 = df_ph[df_ph['PHENOTYPE'] == ph1]
    cohort2 = df_ph[df_ph['PHENOTYPE'] == ph2]
    
    result = logrank_test(
        cohort1['SURVIVAL_YEARS'],
        cohort2['SURVIVAL_YEARS'],
        cohort1['VITAL_STATUS'],
        cohort2['VITAL_STATUS']
    )
    
    name1 = phenotype_names[ph1]
    name2 = phenotype_names[ph2]
    print(f"\n{name1} vs {name2}:")
    print(f"  Test statistic: {result.test_statistic:.3f}")
    print(f"  p-value: {result.p_value:.4f}")
    print(f"  Conclusion: {'Significantly different' if result.p_value < 0.05 else 'Not significantly different'}")

## Survival Summary by Phenotype

In [ ]:
# Survival statistics
kmf = KaplanMeierFitter()

print("\nSurvival Summary Statistics by Phenotype")
print("=" * 70)

for ph in [1, 2, 3]:
    cohort = df_ph[df_ph['PHENOTYPE'] == ph]
    kmf.fit(cohort['SURVIVAL_YEARS'], cohort['VITAL_STATUS'])
    
    # 1-, 3-, 5-year survival
    survival_1y = kmf.survival_function_at_times([1]).values[0] if 1 in kmf.survival_function_.index else np.nan
    survival_3y = kmf.survival_function_at_times([3]).values[0] if 3 in kmf.survival_function_.index else np.nan
    survival_5y = kmf.survival_function_at_times([5]).values[0] if 5 in kmf.survival_function_.index else np.nan
    
    median_survival = kmf.median_survival_time_
    
    name = phenotype_names[ph]
    print(f"\nPhenotype {int(ph)} ({name}, n={len(cohort):,}):")
    print(f"  1-year survival: {survival_1y*100:.1f}%")
    print(f"  3-year survival: {survival_3y*100:.1f}%")
    print(f"  5-year survival: {survival_5y*100:.1f}%")
    print(f"  Median survival: {median_survival:.2f} years")